# 10. Hard Case Mining y Sampling Refinado Thoracolumbar - Colab

Este notebook no cambia la estrategia general del proyecto. Su papel es preparar
mejor la siguiente iteracion del entrenamiento manteniendo el mismo pipeline:

1. localizacion binaria de columna
2. segmentacion multiclase thoracolumbar
3. estimador de ultima vertebra visible
4. clipping anatomico

## Por que existe este notebook

El informe de avance y la bitacora muestran que el cuello de botella real ya no
es encontrar la columna, sino manejar mejor:

- radiografias parciales
- desbalance de clases
- sobreprediccion en la ultima vertebra visible
- casos dificiles de escoliosis

## Objetivo

Construir insumos para una version mas fuerte del pipeline sin romper la
arquitectura actual:

- identificar slices dificiles del dataset
- medir rareza por vertebra y por ultima vertebra visible
- generar pesos refinados de muestreo para `multiclass` y `last_visible`
- exportar listas de casos prioritarios para revision y entrenamiento dirigido

## 0. Preparacion de Colab

Ajusta `PROJECT_ROOT` segun la ubicacion real de la carpeta en tu Google Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# CAMBIA ESTA RUTA SOLO SI TU CARPETA ESTA EN OTRO LUGAR
PROJECT_ROOT = Path("/content/drive/MyDrive/DataRadriografias")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"No existe la carpeta: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/DataRadriografias


## 1. Librerias y resolucion de rutas

Este notebook puede funcionar tanto si tus salidas estan en `analysis_outputs/`
como si quedaron dentro de `reports/analysis_outputs/`.

In [3]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'analysis_outputs' / 'hard_case_mining_thoracolumbar_explained'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def resolve_existing(*relative_candidates: str) -> Path:
    search_roots = [ROOT, ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'No se encontro ninguno de estos archivos: {relative_candidates}')


MANIFEST_PATH = resolve_existing('analysis_outputs/training_manifest_thoracolumbar.csv')
LAST_COMPARE_PATH = resolve_existing(
    'analysis_outputs/last_visible_estimator_thoracolumbar_explained/last_visible_per_sample_compare.csv'
)
LAST_TEST_PRED_PATH = resolve_existing(
    'analysis_outputs/last_visible_estimator_thoracolumbar_explained/last_visible_test_predictions.csv'
)
LAST_SUMMARY_PATH = resolve_existing(
    'analysis_outputs/last_visible_estimator_thoracolumbar_explained/last_visible_experiment_summary.csv'
)
INFERENCE_PER_SAMPLE_PATH = resolve_existing(
    'analysis_outputs/thoracolumbar_inference_analysis_explained/inference_per_sample_summary.csv'
)
POST_V2_COMPARE_PATH = resolve_existing(
    'analysis_outputs/thoracolumbar_postprocess_anatomical_v2_explained/postprocess_v2_per_sample_compare.csv'
)

print('MANIFEST_PATH:', MANIFEST_PATH)
print('LAST_COMPARE_PATH:', LAST_COMPARE_PATH)
print('LAST_TEST_PRED_PATH:', LAST_TEST_PRED_PATH)
print('LAST_SUMMARY_PATH:', LAST_SUMMARY_PATH)
print('INFERENCE_PER_SAMPLE_PATH:', INFERENCE_PER_SAMPLE_PATH)
print('POST_V2_COMPARE_PATH:', POST_V2_COMPARE_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)

MANIFEST_PATH: /content/drive/MyDrive/DataRadriografias/analysis_outputs/training_manifest_thoracolumbar.csv
LAST_COMPARE_PATH: /content/drive/MyDrive/DataRadriografias/analysis_outputs/last_visible_estimator_thoracolumbar_explained/last_visible_per_sample_compare.csv
LAST_TEST_PRED_PATH: /content/drive/MyDrive/DataRadriografias/analysis_outputs/last_visible_estimator_thoracolumbar_explained/last_visible_test_predictions.csv
LAST_SUMMARY_PATH: /content/drive/MyDrive/DataRadriografias/analysis_outputs/last_visible_estimator_thoracolumbar_explained/last_visible_experiment_summary.csv
INFERENCE_PER_SAMPLE_PATH: /content/drive/MyDrive/DataRadriografias/analysis_outputs/thoracolumbar_inference_analysis_explained/inference_per_sample_summary.csv
POST_V2_COMPARE_PATH: /content/drive/MyDrive/DataRadriografias/analysis_outputs/thoracolumbar_postprocess_anatomical_v2_explained/postprocess_v2_per_sample_compare.csv
OUTPUT_DIR: /content/drive/MyDrive/DataRadriografias/analysis_outputs/hard_case_mi

## 2. Carga de tablas base

Reunimos:

- manifiesto thoracolumbar
- errores por muestra del estimador `last_visible`
- resumen por muestra del multiclase crudo
- comparacion por muestra del postproceso `v2`

In [4]:
manifest_df = pd.read_csv(MANIFEST_PATH)
last_compare_df = pd.read_csv(LAST_COMPARE_PATH)
last_test_pred_df = pd.read_csv(LAST_TEST_PRED_PATH)
last_summary_df = pd.read_csv(LAST_SUMMARY_PATH)
inference_per_sample_df = pd.read_csv(INFERENCE_PER_SAMPLE_PATH)
post_v2_compare_df = pd.read_csv(POST_V2_COMPARE_PATH)

present_cols = [
    col for col in manifest_df.columns
    if col.startswith('present_') and col.replace('present_', '')[:1] in {'T', 'L'}
]

manifest_df['is_partial_case'] = manifest_df['num_visible_target_vertebrae'] < 17
manifest_df['is_scoliosis_case'] = manifest_df['split'].astype(str).str.lower().eq('scoliosis')
manifest_df['last_visible_target'] = manifest_df['last_visible_target'].fillna('')

subset_df = manifest_df.loc[
    manifest_df['usable_for_thoracolumbar_partial'].astype(str).str.lower().eq('true')
    | (manifest_df['usable_for_thoracolumbar_partial'] == True)
].copy()

print('manifest_df:', manifest_df.shape)
print('subset_df usable_for_thoracolumbar_partial:', subset_df.shape)
print('last_compare_df:', last_compare_df.shape)
print('last_test_pred_df:', last_test_pred_df.shape)
print('inference_per_sample_df:', inference_per_sample_df.shape)
print('post_v2_compare_df:', post_v2_compare_df.shape)
display(last_summary_df)

manifest_df: (250, 54)
subset_df usable_for_thoracolumbar_partial: (224, 54)
last_compare_df: (45, 21)
last_test_pred_df: (45, 12)
inference_per_sample_df: (45, 15)
post_v2_compare_df: (45, 14)


,metric,value
0,target_subset,partial
1,test_samples,45
2,last_test_exact_acc,0.4
3,last_test_within1_acc,0.5555555555555556
4,last_test_mae,1.8888888888888888
5,last_test_overprediction_rate,0.4666666666666667
6,raw_macro_dice_fg,0.2853897834388499
7,oracle_macro_dice_fg,0.316585840687472
8,last_pred_clip_macro_dice_fg,0.29615453539162606
9,mean_raw_extra_count,3.088888888888889


## 3. Desbalance anatomico en train

Esta seccion responde una pregunta practica:

> Que vertebras y que etiquetas de `last_visible_target` estan menos cubiertas en
> entrenamiento y por tanto deberian pesar mas en el sampler?

In [5]:
train_subset_df = subset_df.loc[
    subset_df['needs_annotation_review'].astype(str).str.lower().ne('true')
    & (subset_df['partition'].astype(str).str.lower() == 'train')
].copy()

presence_rows = []
for col in present_cols:
    label = col.replace('present_', '')
    count = int(pd.to_numeric(train_subset_df[col], errors='coerce').fillna(0).sum())
    presence_rows.append({'vertebra_label': label, 'train_images_with_label': count})

vertebra_presence_train_df = pd.DataFrame(presence_rows).sort_values(
    'train_images_with_label', ascending=True
).reset_index(drop=True)
vertebra_presence_train_df['coverage_ratio'] = (
    vertebra_presence_train_df['train_images_with_label'] / max(len(train_subset_df), 1)
)
vertebra_presence_train_df['rarity_weight'] = (
    vertebra_presence_train_df['train_images_with_label'].max()
    / vertebra_presence_train_df['train_images_with_label'].clip(lower=1)
) ** 0.5

last_visible_distribution_df = (
    train_subset_df.groupby('last_visible_target')
    .size()
    .rename('train_images')
    .reset_index()
    .sort_values('train_images', ascending=True)
    .reset_index(drop=True)
)
last_visible_distribution_df['coverage_ratio'] = (
    last_visible_distribution_df['train_images'] / max(len(train_subset_df), 1)
)
last_visible_distribution_df['rarity_weight'] = (
    last_visible_distribution_df['train_images'].max()
    / last_visible_distribution_df['train_images'].clip(lower=1)
) ** 0.5

display(vertebra_presence_train_df)
display(last_visible_distribution_df)

KeyError: 'partition'

## 4. Union de slices dificiles en test

Aqui conectamos anatomia y errores reales del pipeline actual.

Esto nos permite responder:

- En que tipos de muestra se concentra la sobreprediccion.
- Si los casos parciales y de escoliosis siguen siendo los mas dificiles.
- Si hay targets de `last_visible` especialmente fragiles.

In [ ]:
analysis_df = (
    last_compare_df
    .merge(last_test_pred_df, on='unique_sample_id', how='left', suffixes=('', '_pred'))
    .merge(
        inference_per_sample_df[
            ['unique_sample_id', 'roi_source', 'num_gt_labels', 'num_pred_labels', 'num_missing_labels', 'num_extra_labels']
        ],
        on='unique_sample_id',
        how='left',
    )
    .merge(
        post_v2_compare_df[
            ['unique_sample_id', 'post_missing_count', 'post_extra_count', 'extra_reduction', 'missing_change']
        ],
        on='unique_sample_id',
        how='left',
    )
    .merge(
        manifest_df[
            [
                'unique_sample_id', 'num_visible_target_vertebrae', 'last_visible_target',
                'total_internal_missing_count', 'split', 'has_thoracic', 'has_lumbar'
            ]
        ],
        on='unique_sample_id',
        how='left',
    )
)

analysis_df['is_partial_case'] = analysis_df['num_visible_target_vertebrae'] < 17
analysis_df['is_scoliosis_case'] = analysis_df['split'].astype(str).str.lower().eq('scoliosis')
analysis_df['is_overprediction_case'] = analysis_df['last_overprediction'].fillna(False)
analysis_df['is_underprediction_case'] = analysis_df['last_underprediction'].fillna(False)
analysis_df['benefited_from_last_clip'] = (
    (analysis_df['last_extra_reduction_vs_raw'] > 0)
    & (analysis_df['last_missing_change_vs_raw'] <= 0)
)
analysis_df['harm_from_last_clip'] = analysis_df['last_missing_change_vs_raw'] > 0

display(analysis_df.head())

## 5. Lectura por slices

Agrupamos por caracteristicas clave para detectar donde realmente conviene hacer
mas enfasis de entrenamiento.

In [ ]:
slice_rows = []
slice_specs = [
    ('all_test', pd.Series([True] * len(analysis_df))),
    ('partial', analysis_df['is_partial_case']),
    ('full_range', ~analysis_df['is_partial_case']),
    ('scoliosis', analysis_df['is_scoliosis_case']),
    ('normal', ~analysis_df['is_scoliosis_case']),
    ('internal_missing_gt', analysis_df['total_internal_missing_count'].fillna(0) > 0),
    ('lumbar_visible_gt', analysis_df['last_true_idx'].fillna(-1) >= 12),
    ('thoracic_end_gt', analysis_df['last_true_idx'].fillna(-1) < 12),
]

for slice_name, mask in slice_specs:
    work = analysis_df.loc[mask.fillna(False)].copy()
    if work.empty:
        continue
    slice_rows.append({
        'slice_name': slice_name,
        'images': len(work),
        'exact_acc': float(work['last_exact_match'].mean()),
        'within1_acc': float(work['last_within1_match'].mean()),
        'mae': float(work['last_abs_error'].mean()),
        'overprediction_rate': float(work['last_overprediction'].mean()),
        'mean_raw_extra_count': float(work['raw_extra_count'].mean()),
        'mean_last_extra_count': float(work['last_extra_count'].mean()),
        'mean_last_missing_count': float(work['last_missing_count'].mean()),
        'clean_improvement_rate': float(work['benefited_from_last_clip'].mean()),
        'harm_rate': float(work['harm_from_last_clip'].mean()),
    })

slice_summary_df = pd.DataFrame(slice_rows).sort_values(
    ['clean_improvement_rate', 'within1_acc'], ascending=[False, False]
).reset_index(drop=True)

display(slice_summary_df)

## 6. Ranking de casos dificiles

No todos los errores son igual de utiles para la siguiente iteracion.
Priorizamos los casos donde se combinan:

- error grande en `last_visible`
- sobreprediccion
- mucha etiqueta extra en el multiclase
- escoliosis
- anatomia parcial o irregular

In [ ]:
rarity_map = last_visible_distribution_df.set_index('last_visible_target')['rarity_weight'].to_dict()

hard_case_df = analysis_df.copy()
hard_case_df['rarity_weight'] = hard_case_df['last_visible_target'].map(rarity_map).fillna(1.0)
hard_case_df['hard_case_score'] = (
    hard_case_df['last_abs_error'].fillna(0).astype(float) * 2.0
    + hard_case_df['last_overprediction'].fillna(False).astype(float) * 1.5
    + hard_case_df['raw_extra_count'].fillna(0).astype(float) * 0.6
    + hard_case_df['last_missing_change_vs_raw'].fillna(0).clip(lower=0).astype(float) * 0.8
    + hard_case_df['is_partial_case'].fillna(False).astype(float) * 0.7
    + hard_case_df['is_scoliosis_case'].fillna(False).astype(float) * 0.7
    + hard_case_df['total_internal_missing_count'].fillna(0).astype(float) * 0.3
    + hard_case_df['rarity_weight'].astype(float) * 0.5
)

hard_case_df = hard_case_df.sort_values(
    ['hard_case_score', 'last_abs_error', 'raw_extra_count'],
    ascending=[False, False, False]
).reset_index(drop=True)

display(
    hard_case_df[
        [
            'unique_sample_id', 'split', 'image', 'gt_last_label', 'last_pred_label',
            'last_abs_error', 'last_overprediction', 'raw_extra_count',
            'last_extra_count', 'last_missing_count', 'is_partial_case',
            'is_scoliosis_case', 'total_internal_missing_count', 'hard_case_score'
        ]
    ].head(30)
)

## 7. Sampling refinado para `last_visible`

Esta tabla es la salida mas importante del notebook para entrenamiento.

El peso final combina:

- rareza del target `last_visible_target`
- si la muestra es parcial
- si es escoliosis
- si tiene vacios internos

La idea es usar estos pesos con `WeightedRandomSampler` o como pesos de perdida.

In [ ]:
train_last_df = train_subset_df.copy()
train_last_df['rarity_weight_last_target'] = train_last_df['last_visible_target'].map(rarity_map).fillna(1.0)
train_last_df['partial_weight'] = np.where(train_last_df['num_visible_target_vertebrae'] < 17, 1.20, 1.0)
train_last_df['scoliosis_weight'] = np.where(train_last_df['split'].astype(str).str.lower().eq('scoliosis'), 1.15, 1.0)
train_last_df['internal_missing_weight'] = np.where(train_last_df['total_internal_missing_count'].fillna(0) > 0, 1.10, 1.0)
train_last_df['last_visible_training_weight'] = (
    train_last_df['rarity_weight_last_target']
    * train_last_df['partial_weight']
    * train_last_df['scoliosis_weight']
    * train_last_df['internal_missing_weight']
)
train_last_df['difficulty_bucket'] = pd.cut(
    train_last_df['last_visible_training_weight'],
    bins=[0, 1.10, 1.35, 1.70, np.inf],
    labels=['base', 'medium', 'high', 'very_high'],
    include_lowest=True,
)

last_sampling_weights_df = train_last_df[
    [
        'unique_sample_id', 'split', 'image', 'last_visible_target',
        'num_visible_target_vertebrae', 'total_internal_missing_count',
        'rarity_weight_last_target', 'partial_weight', 'scoliosis_weight',
        'internal_missing_weight', 'last_visible_training_weight', 'difficulty_bucket'
    ]
].sort_values('last_visible_training_weight', ascending=False).reset_index(drop=True)

display(last_sampling_weights_df.head(30))

## 8. Sampling refinado para `multiclass`

Para el modelo multiclase no nos interesa solo la ultima vertebra visible, sino
tambien la rareza de las vertebras presentes dentro de cada muestra.

In [ ]:
vertebra_rarity_map = vertebra_presence_train_df.set_index('vertebra_label')['rarity_weight'].to_dict()

train_multi_df = train_subset_df.copy()

def compute_multiclass_sample_weight(row: pd.Series) -> float:
    present_labels = []
    for col in present_cols:
        value = pd.to_numeric(pd.Series([row[col]]), errors='coerce').fillna(0).iloc[0]
        if int(value) > 0:
            present_labels.append(col.replace('present_', ''))
    if not present_labels:
        return 1.0
    rarity_values = [float(vertebra_rarity_map.get(label, 1.0)) for label in present_labels]
    rarity_component = float(np.mean(rarity_values))
    partial_component = 1.10 if row['num_visible_target_vertebrae'] < 17 else 1.0
    scoliosis_component = 1.10 if str(row['split']).lower() == 'scoliosis' else 1.0
    internal_missing_component = 1.08 if float(row.get('total_internal_missing_count', 0) or 0) > 0 else 1.0
    return rarity_component * partial_component * scoliosis_component * internal_missing_component


train_multi_df['multiclass_training_weight'] = train_multi_df.apply(compute_multiclass_sample_weight, axis=1)
train_multi_df['multiclass_weight_bucket'] = pd.cut(
    train_multi_df['multiclass_training_weight'],
    bins=[0, 1.08, 1.28, 1.55, np.inf],
    labels=['base', 'medium', 'high', 'very_high'],
    include_lowest=True,
)

multiclass_sampling_weights_df = train_multi_df[
    [
        'unique_sample_id', 'split', 'image', 'num_visible_target_vertebrae',
        'last_visible_target', 'total_internal_missing_count',
        'multiclass_training_weight', 'multiclass_weight_bucket'
    ]
].sort_values('multiclass_training_weight', ascending=False).reset_index(drop=True)

display(multiclass_sampling_weights_df.head(30))

## 9. Recomendaciones concretas para la siguiente iteracion

Esta tabla traduce el analisis en acciones tecnicas claras.

In [ ]:
recommendations_df = pd.DataFrame([
    {
        'priority': 1,
        'focus': 'last_visible estimator v2',
        'recommendation': 'Usar WeightedRandomSampler con `last_visible_training_weight` y penalizar mas la sobreprediccion que la subprediccion.',
        'reason': 'El error dominante sigue siendo extender la anatomia mas alla de lo visible.',
    },
    {
        'priority': 2,
        'focus': 'multiclass thoracolumbar',
        'recommendation': 'Reentrenar con `multiclass_training_weight` para aumentar exposicion a vertebras raras y casos parciales.',
        'reason': 'El bloque toracico y ciertos finales cortos siguen subrepresentados.',
    },
    {
        'priority': 3,
        'focus': 'hard case curriculum',
        'recommendation': 'Reservar un subconjunto de casos duros para validacion cualitativa obligatoria despues de cada corrida.',
        'reason': 'Las metricas globales no muestran por si solas el problema de sobreprediccion anatomica.',
    },
    {
        'priority': 4,
        'focus': 'annotation review',
        'recommendation': 'Revisar primero las imagenes con `needs_annotation_review` y luego reincorporarlas si quedan consistentes.',
        'reason': 'La calidad de anotacion limita el techo del pipeline completo.',
    },
]).sort_values('priority')

display(recommendations_df)

## 10. Visualizaciones rapidas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(
    vertebra_presence_train_df['vertebra_label'],
    vertebra_presence_train_df['train_images_with_label'],
    color='#2f6db3'
)
axes[0].set_title('Cobertura train por vertebra')
axes[0].tick_params(axis='x', rotation=90)
axes[0].grid(alpha=0.25, axis='y')

axes[1].bar(
    last_visible_distribution_df['last_visible_target'],
    last_visible_distribution_df['train_images'],
    color='#c46b2d'
)
axes[1].set_title('Distribucion train de last_visible_target')
axes[1].tick_params(axis='x', rotation=90)
axes[1].grid(alpha=0.25, axis='y')

plt.tight_layout()
plt.show()

## 11. Exportacion de resultados

Estas salidas quedan listas para ser usadas por un notebook futuro de
reentrenamiento.

In [ ]:
vertebra_presence_path = OUTPUT_DIR / 'vertebra_presence_train_imbalance.csv'
last_distribution_path = OUTPUT_DIR / 'last_visible_target_train_distribution.csv'
slice_summary_path = OUTPUT_DIR / 'last_visible_error_slice_summary.csv'
hard_cases_path = OUTPUT_DIR / 'hard_cases_last_visible_test.csv'
last_sampling_path = OUTPUT_DIR / 'train_sampling_weights_last_visible.csv'
multiclass_sampling_path = OUTPUT_DIR / 'train_sampling_weights_multiclass.csv'
recommendations_path = OUTPUT_DIR / 'refined_sampling_recommendations.csv'

vertebra_presence_train_df.to_csv(vertebra_presence_path, index=False)
last_visible_distribution_df.to_csv(last_distribution_path, index=False)
slice_summary_df.to_csv(slice_summary_path, index=False)
hard_case_df.to_csv(hard_cases_path, index=False)
last_sampling_weights_df.to_csv(last_sampling_path, index=False)
multiclass_sampling_weights_df.to_csv(multiclass_sampling_path, index=False)
recommendations_df.to_csv(recommendations_path, index=False)

print('Guardado:', vertebra_presence_path)
print('Guardado:', last_distribution_path)
print('Guardado:', slice_summary_path)
print('Guardado:', hard_cases_path)
print('Guardado:', last_sampling_path)
print('Guardado:', multiclass_sampling_path)
print('Guardado:', recommendations_path)